# Data Cleaning

Collect or select meaningful data. Clean, merge, transform, and prepare the dataset with Python.

In [2]:
#@title Download and unzip the dataset from Google Drive
%pip install -q gdown
import gdown, zipfile, pathlib

gdown.download(id="1kmS0LCIPbgt_c_sQxQ8eGz1A9MJhj5_G", output="dataset.zip")

with zipfile.ZipFile("dataset.zip") as z:
    z.extractall("data")
BASE = pathlib.Path("data/Gapminder Dataset + Proposal")
OTHER, DEATHS = BASE / "other", BASE / "Deaths Datasets"
print("other/:", len(list(OTHER.glob('*.csv'))), "files | Deaths Datasets/:", len(list(DEATHS.glob('*.csv'))), "files")


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


Downloading...
From: https://drive.google.com/uc?id=1kmS0LCIPbgt_c_sQxQ8eGz1A9MJhj5_G
To: c:\Users\delyn\Downloads\dataset.zip
100%|██████████| 2.90M/2.90M [00:00<00:00, 6.65MB/s]


other/: 18 files | Deaths Datasets/: 13 files


In [3]:
#@title Reshape the wide Gapminder files into their long format
import pandas as pd

WIDE = {
 "life_expectancy.csv": "life_expectancy_years",
 "child_mortality_0_5_year_olds_dying_per_1000_born.csv": "child_mortality_age0_5_per_1000_born",
 "sex_ratio_all_age_groups.csv": "sex_ratio_all_ages",
 "total_health_spending_per_person_us.csv": "health_spending_per_person_usd",
 "military_expenditure_percent_of_gdp.csv": "military_expenditure_pct_gdp",
 "gdp.csv": "gdp_per_capita_ppp_usd",
 "urban_population_percent_of_total.csv": "urban_population_pct",
}
YEARS = [str(y) for y in range(2010, 2023)]  # 2010-2022
POP_FROM_WIDE = (OTHER / "population.csv").exists()
if POP_FROM_WIDE:
    WIDE["population.csv"] = "population_total"
else:
    print("population.csv not in this zip: population_total falls back to Gabriel's DDF file (annual only to 2013)")

frames, name2geo, present = [], {}, {}
# water only exists in this zip as a DDF file; its country set counts for the strict filter too
for ddf_name in ["at_least_basic_water_source_overall_access_percent"] + ([] if POP_FROM_WIDE else ["total_population_with_projections"]):
    _d = pd.read_csv(OTHER / f"ddf--datapoints--{ddf_name}--by--geo--time.csv", usecols=["geo"])
    present[ddf_name] = set(_d["geo"])
for fname, col in WIDE.items():
    df = pd.read_csv(OTHER / fname, encoding="utf-8-sig")
    present[fname] = set(df["geo"])
    for n, g in zip(df["name"], df["geo"]): name2geo[n] = g
    long = df[["geo"] + YEARS].melt(id_vars="geo", var_name="year", value_name=col)
    long["year"] = long["year"].astype(int)
    long = long.rename(columns={"geo": "country_code"})
    long["country_code"] = long["country_code"].str.upper()
    frames.append(long.set_index(["country_code", "year"]))
wide = pd.concat(frames, axis=1).reset_index()
wide["country"] = wide["country_code"].map({g.upper(): n for n, g in name2geo.items()})
print("wide panel:", wide.shape)

wide panel: (2535, 11)


In [4]:
#@title Stack DDF files
import re, glob

RENAME = {
 "gdppercapita_us_inflation_adjusted": "gdp_per_capita_usd_inflation_adjusted",
 "government_health_spending_per_person_us": "govt_health_spending_per_person_usd",
 "life_expectancy_at_birth_data_from_ihme": "life_expectancy_ihme_years",
 "all_causes_deaths_in_newborn_per_1000_births": "newborn_deaths_per_1000_births",
 "urban_population": "urban_population_count",
 "at_least_basic_water_source_overall_access_percent": "basic_water_access_pct",
 "total_population_with_projections": "population_total",
}
DROP_DDF = {"epidemic_deaths_annual_number", "malaria_number_of_deaths_reported",  # end 2008 / 2006, no 2010+ data
            "male_population_with_projections", "female_population_with_projections",  # sex ratio is already a wide file
            "life_expectancy_at_birth_data_from_ihme"}  # second life expectancy series, ends 2013; the project uses life_expectancy_years

if POP_FROM_WIDE:
    DROP_DDF.add("total_population_with_projections")  # population_total now comes from the wide population.csv

frames = []
for path in sorted(glob.glob(str(OTHER / "ddf*.csv")) + glob.glob(str(DEATHS / "ddf*.csv"))):
    raw = re.search(r"ddf--datapoints--(.+?)--by--geo--time\.csv", path).group(1)
    if raw in DROP_DDF:
        print("dropping:", raw); continue
    df = pd.read_csv(path)
    vcol = [c for c in df.columns if c not in ("geo", "time")][0]
    df = df.rename(columns={"geo": "country_code", "time": "year", vcol: RENAME.get(raw, raw)})
    df["country_code"] = df["country_code"].str.upper()
    frames.append(df.set_index(["country_code", "year"]))
ddf = pd.concat(frames, axis=1).reset_index()
ddf = ddf[(ddf["year"] >= 2010) & (ddf["year"] <= 2022)]
print("ddf extras:", ddf.shape)

dropping: epidemic_deaths_annual_number
dropping: malaria_number_of_deaths_reported
dropping: female_population_with_projections
dropping: life_expectancy_at_birth_data_from_ihme
dropping: male_population_with_projections
dropping: total_population_with_projections
ddf extras: (2899, 18)


In [5]:
#@title Switches the GBD causes of death file from long to wide and unifies country names

import csv

GBD_MAP = {"Bolivia (Plurinational State of)": "Bolivia", "Brunei Darussalam": "Brunei", "Cabo Verde": "Cape Verde", "Congo": "Congo, Rep.", "Czechia": "Czech Republic", "Côte d'Ivoire": "Cote d'Ivoire", "Democratic People's Republic of Korea": "North Korea", "Democratic Republic of the Congo": "Congo, Dem. Rep.", "Iran (Islamic Republic of)": "Iran", "Kyrgyzstan": "Kyrgyz Republic", "Lao People's Democratic Republic": "Lao", "Micronesia (Federated States of)": "Micronesia, Fed. Sts.", "Republic of Korea": "South Korea", "Republic of Moldova": "Moldova", "Russian Federation": "Russia", "Saint Kitts and Nevis": "St. Kitts and Nevis", "Saint Lucia": "St. Lucia", "Saint Vincent and the Grenadines": "St. Vincent and the Grenadines", "Slovakia": "Slovak Republic", "Syrian Arab Republic": "Syria", "Türkiye": "Turkey", "United Arab Emirates": "UAE", "United Kingdom": "UK", "United Republic of Tanzania": "Tanzania", "United States of America": "USA", "Venezuela (Bolivarian Republic of)": "Venezuela", "Viet Nam": "Vietnam"}
rows = []
with open(DEATHS / "IHME GBD Causes of death.csv", encoding="utf-8") as fh:
    for r in csv.DictReader(fh):
        y = int(r["year"])
        if not (2010 <= y <= 2022): continue
        geo = name2geo.get(GBD_MAP.get(r["location"], r["location"]))
        if geo is None: continue  # GBD territory with no Gapminder match
        cause = "death_share_" + re.sub(r"[^a-z0-9]+", "_", r["cause"].lower()).strip("_") + "_pct"
        rows.append({"country_code": geo.upper(), "year": y, "cause": cause,
                     "val": float(r["val"].strip().replace("%", ""))})
gbd = pd.DataFrame(rows).pivot_table(index=["country_code", "year"], columns="cause",
                                     values="val", aggfunc="first").reset_index()
print("gbd wide:", gbd.shape)

gbd wide: (2483, 23)


In [6]:
#@title Merges everything into the final cleaned table


# --- Merge everything into the final cleaned table ---
from IPython.display import display

df = wide.merge(gbd, on=["country_code", "year"], how="inner").merge(ddf, on=["country_code", "year"], how="left")
strict_codes = set.intersection(*present.values()) & set(gbd["country_code"].str.lower())
df = df[df["country_code"].str.lower().isin(strict_codes)]

# Country names first, sorted by name then year
first_cols = ["country", "year", "country_code"]
df = df[first_cols + [c for c in df.columns if c not in first_cols]].sort_values(["country", "year"]).reset_index(drop=True)

print("FINAL:", df.shape, "| countries:", df["country"].nunique(),
      "| years:", df["year"].min(), "-", df["year"].max())
print("duplicates:", df.duplicated(["country_code", "year"]).sum())
df.to_csv("health_country_year_2010_2022_combined.csv", index=False)

# --- Preview: key columns and first 10 rows only, "..." marks that there is more ---
preview_cols = ["country", "year", "life_expectancy_years", "child_mortality_age0_5_per_1000_born",
                "gdp_per_capita_ppp_usd", "health_spending_per_person_usd",
                "basic_water_access_pct", "urban_population_pct"]
preview = df[preview_cols].head(10).astype(object)   # display copy only, df itself is not changed
preview["..."] = "..."                               # more columns to the right
dots_row = pd.DataFrame([["..."] * len(preview.columns)], columns=preview.columns)
preview = pd.concat([preview, dots_row], ignore_index=True)   # more rows below
display(preview)
print(f"Preview only: 10 of {len(df)} rows and {len(preview_cols)} of {len(df.columns)} columns. "
      f"The full table is the dataframe df and the saved CSV.")

# --- Summary: what each column is, and what is missing ---
DESCRIBE = {
 "country": "Country name",
 "year": "Year",
 "country_code": "ISO 3-letter country code",
 "life_expectancy_years": "Life expectancy at birth (years)",
 "child_mortality_age0_5_per_1000_born": "Children dying before age 5, per 1,000 born",
 "sex_ratio_all_ages": "Males per 100 females",
 "health_spending_per_person_usd": "Total health spending per person (USD)",
 "military_expenditure_pct_gdp": "Military spending (% of GDP)",
 "gdp_per_capita_ppp_usd": "GDP per capita, PPP (USD)",
 "urban_population_pct": "Urban population (% of total)",
 "population_total": "Total population",
 "newborn_deaths_per_1000_births": "Newborn deaths per 1,000 births",
 "basic_water_access_pct": "% of people with at least basic drinking water",
 "gdp_per_capita_usd_inflation_adjusted": "GDP per capita, inflation-adjusted USD (second GDP series)",
 "govt_health_spending_per_person_usd": "Government health spending per person (USD)",
 "urban_population_count": "Urban population (number of people)",
 "urban_population_growth_annual_percent": "Urban population growth (% per year)",
}

def describe_col(c):
    if c.startswith("death_share_"):   # the 21 GBD columns describe themselves
        cause = c[len("death_share_"):-len("_pct")].replace("_", " ")
        cause = cause.replace("hiv aids", "HIV/AIDS").replace("non communicable", "non-communicable")
        return f"Share of all deaths from {cause} (%)"
    if "cancer" in c:                  # the 10 cancer count columns
        base = c.replace("_number_of_female_deaths", "").replace("_number_of_male_deaths", "")
        base = base.replace("colonandrectum", "colon and rectum").replace("_", " ")
        sex = "female" if "female" in c else "male"
        return f"{base.capitalize()} deaths, {sex} (count)"
    return DESCRIBE.get(c, "")

summary = pd.DataFrame({
    "what_it_is": [describe_col(c) for c in df.columns],
    "missing": df.isna().sum(),
})
summary["missing_pct"] = (summary["missing"] / len(df) * 100).round(1)
display(summary)

FINAL: (2080, 48) | countries: 160 | years: 2010 - 2022
duplicates: 0


,country,year,life_expectancy_years,child_mortality_age0_5_per_1000_born,gdp_per_capita_ppp_usd,health_spending_per_person_usd,basic_water_access_pct,urban_population_pct,...
0,Afghanistan,2010,65.6,87.76,2848.58606,46.42439,49.13379,22.26148,...
1,Afghanistan,2011,66.0,83.97,2757.05259,52.18722,51.38967,22.82057,...
2,Afghanistan,2012,66.3,80.34,2985.31914,52.45247,53.65581,23.34344,...
3,Afghanistan,2013,66.7,76.87,3046.57993,56.16043,55.93214,23.80771,...
4,Afghanistan,2014,66.5,73.6,3017.94254,60.04951,58.21857,24.19102,...
5,Afghanistan,2015,66.3,70.45,2967.69207,59.91961,60.51535,24.46491,...
6,Afghanistan,2016,66.5,67.49,2958.7854,61.37234,62.82207,24.65883,...
7,Afghanistan,2017,66.6,64.83,2952.99892,66.82388,65.14214,24.83528,...
8,Afghanistan,2018,66.3,62.34,2902.39211,71.22509,67.47587,24.99916,...
9,Afghanistan,2019,66.2,60.14,2927.24514,74.06424,69.8227,25.14373,...


Preview only: 10 of 2080 rows and 8 of 48 columns. The full table is the dataframe df and the saved CSV.


,what_it_is,missing,missing_pct
country,Country name,0,0.0
year,Year,0,0.0
country_code,ISO 3-letter country code,0,0.0
life_expectancy_years,Life expectancy at birth (years),0,0.0
child_mortality_age0_5_per_1000_born,"Children dying before age 5, per 1,000 born",0,0.0
sex_ratio_all_ages,Males per 100 females,0,0.0
health_spending_per_person_usd,Total health spending per person (USD),22,1.1
military_expenditure_pct_gdp,Military spending (% of GDP),165,7.9
gdp_per_capita_ppp_usd,"GDP per capita, PPP (USD)",0,0.0
urban_population_pct,Urban population (% of total),0,0.0


# Analysis & Insights

**From Canvas Final project:**

Ask focused questions and explore your dataset using methods such as:

summary statistics, trends, distributions

comparisons across groups or time

correlations or relationships

segmenting or filtering to reveal patterns

Your goal is to move beyond description and provide meaningful and interpretable insights.

# Interactive Visualizations

In [8]:
#@title Interactive GDP vs Life Expectancy bubble chart (Altair)
%pip install -q altair
import pandas as pd
import altair as alt

# load directly from cleaned CSV so cell is self-contained
df = pd.read_csv("health_country_year_2010_2022_combined.csv")
alt.data_transformers.disable_max_rows()
# year slider
year_slider = alt.binding_range(
    min=int(df["year"].min()), max=int(df["year"].max()), step=1, name="Year: "
)
year_select=alt.selection_point(
    fields=["year"], bind=year_slider, value=int(df["year"].min())
)
# Click-to-highlight a continent/region if you add a region column later;
# for now this highlights on hover via tooltip + opacity
hover = alt.selection_point(fields=["country"], on="mouseover", empty=False)
chart=(
    alt.Chart(df)
    .mark_circle()
    .encode(
        x=alt.X(
            "gdp_per_capita_ppp_usd:Q",
            scale=alt.Scale(type="log"),
            title="GDP per capita (PPP, USD, log scale)",
        ),
        y=alt.Y("life_expectancy_years:Q", title="Life expectancy (years)",
                scale=alt.Scale(domain=[30, 90])),
        size=alt.Size(
            "population_total:Q",
            title="Population",
            scale=alt.Scale(range=[20, 1200]),
        ),
        color=alt.Color(
            "health_spending_per_person_usd:Q",
            title="Health spending / person (USD)",
            scale=alt.Scale(scheme="viridis"),
        ),
        opacity=alt.condition(hover, alt.value(1.0), alt.value(0.65)),
        tooltip=[
            alt.Tooltip("country:N", title="Country"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("gdp_per_capita_ppp_usd:Q", title="GDP per capita", format=",.0f"),
            alt.Tooltip("life_expectancy_years:Q", title="Life expectancy", format=".1f"),
            alt.Tooltip("health_spending_per_person_usd:Q", title="Health spending/person", format=",.0f"),
            alt.Tooltip("population_total:Q", title="Population", format=",.0f"),
        ],
    )
    .add_params(year_select, hover)
    .transform_filter(year_select)
    .properties(
        width=700,
        height=460,
        title="GDP vs. Life Expectancy — drag the slider to change year "
              "(bubble size = population, color = health spending per person)",
    )
    .interactive()
)
chart


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


alt.Chart(...)

# We also need:

**README.MD**

**Interactive Dashboard**

**Unlisted YT video or In class presentation**

# Interactive Dashboard (and other cool visualization stuff)

In [9]:
%pip install streamlit

   ---------------------------------------- 0.0/11.2 MB ? eta -:--:--
   --- ------------------------------------ 1.0/11.2 MB 5.2 MB/s eta 0:00:02
   ------ --------------------------------- 1.8/11.2 MB 5.1 MB/s eta 0:00:02
   ----------- ---------------------------- 3.1/11.2 MB 5.2 MB/s eta 0:00:02
   -------------- ------------------------- 4.2/11.2 MB 5.2 MB/s eta 0:00:02
   ------------------ --------------------- 5.2/11.2 MB 5.2 MB/s eta 0:00:02
   ----------------------- ---------------- 6.6/11.2 MB 5.2 MB/s eta 0:00:01
   --------------------------- ------------ 7.6/11.2 MB 5.2 MB/s eta 0:00:01
   ------------------------------ --------- 8.7/11.2 MB 5.2 MB/s eta 0:00:01
   ---------------------------------- ----- 9.7/11.2 MB 5.2 MB/s eta 0:00:01
   ---------------------------------------  11.0/11.2 MB 5.2 MB/s eta 0:00:01
   ---------------------------------------- 11.2/11.2 MB 5.1 MB/s  0:00:02
   ---------------------------------------- 0.0/7.3 MB ? eta -:--:--
   ----- ------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import streamlit as st

st.title("Gapminder Data Analysis")

st.dataframe(df)

2026-11-24 10:16:30.818 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-24 10:16:31.541 
  command:

    streamlit run C:\Users\delyn\AppData\Roaming\Python\Python315\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-11-24 10:16:31.543 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-24 10:16:31.544 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-24 10:16:31.890 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-24 10:16:31.892 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-24 10:16:31.893 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-24 10:16:31.895 Thre

DeltaGenerator()

In [11]:
st.altair_chart(chart, use_container_width=True)

2026-11-24 10:19:00.982 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'` or specify an integer width.
2026-11-24 10:19:00.984 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-24 10:19:00.985 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-24 10:19:00.986 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()